In [1]:
import torch
import torch.nn as nn
import math
import numpy as np
import pandas as pd
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
class MixedDimensionEmbedding(nn.Module):
    def __init__(self, block_vocab_sizes, base_dim, alpha=0.5):
        super(MixedDimensionEmbedding, self).__init__()
        self.num_blocks = len(block_vocab_sizes)
        self.base_dim = base_dim
        
        self.embeddings = nn.ModuleList()
        self.projections = nn.ModuleList()
        
        # Tìm vocab_size nhỏ nhất (tương ứng với xác suất truy vấn cao nhất)
        v_min = float(min(block_vocab_sizes))
        
        for vocab_size in block_vocab_sizes:
            # 1. Tính số chiều tỷ lệ NGHỊCH với vocab_size (chuẩn theo paper Facebook)
            raw_dim = base_dim * ((v_min / float(vocab_size)) ** alpha)
            
            # Làm tròn về lũy thừa của 2 gần nhất (1, 2, 4, 8, 16, 32, 64)
            dim = max(1, 2 ** round(math.log2(max(raw_dim, 1e-9))))
            dim = min(dim, base_dim)
            emb_dim = int(dim)
            
            print(f"Vocab: {vocab_size} -> Dim: {emb_dim}")
            
            # 2. Khởi tạo Embedding
            self.embeddings.append(nn.Embedding(vocab_size, emb_dim))
            
            # 3. Khởi tạo Linear Projection để đưa về base_dim
            if emb_dim != base_dim:
                self.projections.append(nn.Linear(emb_dim, base_dim, bias=False))
            else:
                self.projections.append(nn.Identity())
                
    def forward(self, cat_x):
        # cat_x có shape: (batch_size, num_blocks)
        projected_embs = []
        
        # Lặp qua từng cột (từng feature)
        for i in range(self.num_blocks):
            # Lấy toàn bộ batch của feature thứ i
            feature_col = cat_x[:, i] 
            
            # Đi qua Embedding -> (batch_size, emb_dim)
            emb = self.embeddings[i](feature_col)
            
            # Đi qua Projection -> (batch_size, base_dim)
            proj = self.projections[i](emb)
            
            projected_embs.append(proj)
            
        # Ghép nối tất cả dọc theo chiều feature -> (batch_size, num_blocks * base_dim)
        return torch.cat(projected_embs, dim=1)

In [3]:
class CrossNetwork(nn.Module):
    def __init__(self, input_dim, num_layers): 
        super(CrossNetwork, self).__init__()
        self.num_layers = num_layers
        self.cross_weights = nn.ParameterList([
            nn.Parameter(torch.randn(input_dim, 1) * 0.01) for _ in range(num_layers)
        ])
        self.cross_bias = nn.ParameterList([
            nn.Parameter(torch.zeros(input_dim)) for _ in range(num_layers)
        ])

    def forward(self, x_0):
        x_l = x_0
        for i in range(self.num_layers):
            xl_w = torch.matmul(x_l, self.cross_weights[i])
            x_l = x_0 * xl_w + self.cross_bias[i] + x_l
        return x_l


class DeepCrossNetworkMDE(nn.Module):
    def __init__(self, block_vocab_sizes, num_dense_features, base_dim=64, alpha=0.5):
        super(DeepCrossNetworkMDE, self).__init__()
        self.base_dim = base_dim
        self.num_cat_features = len(block_vocab_sizes)
        
        # 1. Khởi tạo Mixed Dimension Embedding
        self.mde = MixedDimensionEmbedding(
            block_vocab_sizes=block_vocab_sizes,
            base_dim=base_dim,
            alpha=alpha
        )
        
        # Tổng số chiều đầu vào = Dense + (Categorical * base_dim)
        input_dim = num_dense_features + (self.num_cat_features * base_dim)
        
        # 2. Mạng Cross
        self.cross_net = CrossNetwork(input_dim, num_layers=3) 
        
        # 3. Mạng Deep
        self.deep_net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2)
        )
        
        # 4. Lớp Output (Đầu ra Cross là input_dim, đầu ra Deep là 64)
        self.out_layer = nn.Linear(input_dim + 64, 1)

    def forward(self, dense_x, cat_x):
        # 1. Lấy embedding đã được quy chuẩn về base_dim
        # Đầu ra có shape: (batch_size, num_cat_features * base_dim)
        cat_embs = self.mde(cat_x)
        
        # 2. Ghép nối Dense và Categorical Embs
        x_stacked = torch.cat([dense_x, cat_embs], dim=1)
            
        # 3. Đưa qua Cross và Deep
        cross_out = self.cross_net(x_stacked)
        deep_out = self.deep_net(x_stacked)
        
        # 4. Ghép nối kết quả và dự đoán
        combined = torch.cat([cross_out, deep_out], dim=1)
        logits = self.out_layer(combined)
        
        # Ép về 1D để dùng với BCEWithLogitsLoss
        return logits.squeeze(1)

In [4]:
import math

class StandardCriteoDataset(Dataset):
    def __init__(self, hf_dataset, dense_cols, cat_cols):
        self.data = hf_dataset
        self.dense_cols = dense_cols
        self.cat_cols = cat_cols
        
        self.block_vocab_sizes = []
        self.cat_mappers = []
        
        print("Đang đếm Vocabulary Size cho 26 khối Categorical...")
        self._build_vocab()
        
    def _build_vocab(self):
        df = self.data.to_pandas()
        for col in tqdm(self.cat_cols, desc="Building Vocab"):
            unique_vals = df[col].dropna().unique()
            mapper = {val: idx + 1 for idx, val in enumerate(unique_vals)}
            self.cat_mappers.append(mapper)
            self.block_vocab_sizes.append(len(unique_vals) + 1)
            
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        
        dense_x = []
        for c in self.dense_cols:
            val = row[c]
            # Nếu giá trị là None hoặc số âm, gán bằng 0
            if val is None or val < 0:
                dense_x.append(0.0)
            else:
                # Công thức chuẩn hóa: log(1 + giá_trị)
                dense_x.append(math.log1p(float(val))) 
        
        # Categorical
        cat_ids = []
        for col_idx, col_name in enumerate(self.cat_cols):
            val = row[col_name]
            mapper = self.cat_mappers[col_idx]
            cat_ids.append(mapper.get(val, 0)) 
            
        label = float(row['label'])

        return (
            torch.tensor(dense_x, dtype=torch.float32),
            torch.tensor(cat_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32)
        )

In [5]:
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, random_split
import glob
import os

# =========================
# 1. Khai báo cột
# =========================
dense_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

# =========================
# 2. Load parquet local
# =========================
print("Đang tải dữ liệu parquet local...")

# đường dẫn folder chứa các file parquet
data_dir = "/kaggle/input/datasets/huy291/criteo-cleaned-data/data"

# lấy toàn bộ parquet
parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))

print(f"Số file parquet: {len(parquet_files)}")

# load bằng HuggingFace datasets
raw_dataset = load_dataset(
    "parquet",
    data_files=parquet_files,
    split="train"
)

print(raw_dataset)

# =========================
# 3. Dataset wrapper
# =========================
dataset = StandardCriteoDataset(
    raw_dataset,
    dense_cols,
    cat_cols
)

# =========================
# 4. Chia train/val
# =========================
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print(f"Tổng số mẫu: {len(dataset)}")
print(f"Train: {train_size}")
print(f"Val: {val_size}")

# =========================
# 5. DataLoader
# =========================
train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Hoàn tất tạo DataLoader.")

Đang tải dữ liệu parquet local...
Số file parquet: 50


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/34 [00:00<?, ?it/s]

Dataset({
    features: ['label', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'I10', 'I11', 'I12', 'I13', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21', 'C22', 'C23', 'C24', 'C25', 'C26'],
    num_rows: 45840617
})
Đang đếm Vocabulary Size cho 26 khối Categorical...


Building Vocab:   0%|          | 0/26 [00:00<?, ?it/s]

Tổng số mẫu: 45840617
Train: 41256555
Val: 4584062
Hoàn tất tạo DataLoader.


In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {num_gpus}")
for i in range(num_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

model = DeepCrossNetworkMDE(
    block_vocab_sizes=dataset.block_vocab_sizes,
    num_dense_features=len(dense_cols),
    base_dim=64,
    alpha=0.5
)

# Wrap DataParallel nếu có >= 2 GPU
if num_gpus > 1:
    print(f"Dùng DataParallel trên {num_gpus} GPU")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

print("Bắt đầu huấn luyện...")
EPOCHS = 3

for epoch in range(EPOCHS):
    # ---------- TRAIN ----------
    model.train()
    train_loss = 0.0
    train_targets = []
    train_preds = []

    train_bar = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{EPOCHS}")

    for dense_x, cat_x, labels in train_bar:
        # pin_memory + non_blocking giúp transfer CPU→GPU nhanh hơn
        dense_x = dense_x.to(device, non_blocking=True)
        cat_x   = cat_x.to(device,   non_blocking=True)
        labels  = labels.to(device,   non_blocking=True)

        optimizer.zero_grad()
        logits = model(dense_x, cat_x)

        # DataParallel trả về output ghép từ nhiều GPU → shape vẫn đúng
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        # .mean() để lấy scalar khi DataParallel trả về tensor nhiều phần tử
        train_loss += loss.item()

        probs = torch.sigmoid(logits)
        train_targets.extend(labels.detach().cpu().numpy())
        train_preds.extend(probs.detach().cpu().numpy())

        train_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_auc = roc_auc_score(train_targets, train_preds)

    # ---------- VALIDATION ----------
    model.eval()
    val_loss = 0.0
    val_targets = []
    val_preds = []

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"[VAL] Epoch {epoch+1}/{EPOCHS}", leave=False)
        for dense_x, cat_x, labels in val_bar:
            dense_x = dense_x.to(device, non_blocking=True)
            cat_x   = cat_x.to(device,   non_blocking=True)
            labels  = labels.to(device,   non_blocking=True)

            logits = model(dense_x, cat_x)
            loss   = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            val_targets.extend(labels.cpu().numpy())
            val_preds.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_auc = roc_auc_score(val_targets, val_preds)

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss: {avg_train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {avg_val_loss:.4f} | Val AUC: {val_auc:.4f}\n"
    )

Số GPU khả dụng: 1
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition
Vocab: 1461 -> Dim: 4
Vocab: 584 -> Dim: 4
Vocab: 10131228 -> Dim: 1
Vocab: 2202609 -> Dim: 1
Vocab: 306 -> Dim: 8
Vocab: 25 -> Dim: 32
Vocab: 12518 -> Dim: 1
Vocab: 634 -> Dim: 4
Vocab: 4 -> Dim: 64
Vocab: 93146 -> Dim: 1
Vocab: 5684 -> Dim: 2
Vocab: 8351594 -> Dim: 1
Vocab: 3195 -> Dim: 2
Vocab: 28 -> Dim: 32
Vocab: 14993 -> Dim: 1
Vocab: 5461307 -> Dim: 1
Vocab: 11 -> Dim: 32
Vocab: 5653 -> Dim: 2
Vocab: 2174 -> Dim: 2
Vocab: 5 -> Dim: 64
Vocab: 7046548 -> Dim: 1
Vocab: 19 -> Dim: 32
Vocab: 16 -> Dim: 32
Vocab: 286182 -> Dim: 1
Vocab: 106 -> Dim: 16
Vocab: 142573 -> Dim: 1
Bắt đầu huấn luyện...


[TRAIN] Epoch 1/3: 100%|██████████| 10073/10073 [22:54<00:00,  7.33it/s, Loss=0.4467]


Epoch 1: Train Loss: 0.4711 | Train AUC: 0.7814 || Val Loss: 0.4549 | Val AUC: 0.7965



[TRAIN] Epoch 2/3: 100%|██████████| 10073/10073 [22:57<00:00,  7.31it/s, Loss=0.4648] 


Epoch 2: Train Loss: 4.3819 | Train AUC: 0.7671 || Val Loss: 0.4626 | Val AUC: 0.7870



[TRAIN] Epoch 3/3: 100%|██████████| 10073/10073 [22:57<00:00,  7.31it/s, Loss=0.4333]


Epoch 3: Train Loss: 0.4024 | Train AUC: 0.8453 || Val Loss: 0.4810 | Val AUC: 0.7722

